# 07 - Final Portfolio Validation / ???????

This notebook is the final validation runbook for the portfolio-ready project. It does not retrain YOLO. It verifies existing Drive artifacts, latest package code, report validation/fallback, single-image report safety, compact demo display helpers, LLM/RAG eval, and lightweight evidence backup.

? notebook ????????????????? YOLO?????? Drive ?????????????/????????????? demo ?? helper?LLM/RAG eval ????????

## Scope / ??

- Uses existing Drive artifacts under `/content/drive/MyDrive/CarDD_YOLO11`.
- Does not delete datasets, weights, adapters, ONNX exports, runs, or backups.
- Uses package-level validators; notebook cells only orchestrate and assert.
- Keeps full `mask_polygon` in debug JSON only, not in public summary/table.

- ?? Drive ??????
- ?????????adapter?ONNX?runs ????
- ???? validator?notebook ?????????
- ?? `mask_polygon` ???? debug JSON?????????/???

In [ ]:
# CN: ?? Drive ????????
# EN: Mount Drive and configure shared paths.
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
REPO_URL = 'https://github.com/lsdlBlueDragon/ai-powered-vehicle-damage-assessment-pipeline.git'
REPO_CLONE_DIR = Path('/content/ai-powered-vehicle-damage-assessment-pipeline')

YOLO_WEIGHTS = DRIVE_ROOT / 'runs/train/yolo11n_seg/weights/best.pt'
QWEN_ADAPTER_DIR = DRIVE_ROOT / 'llm_adapters/qwen2_5_7b_cardd_report_lora'
CONTEXT_JSON = DRIVE_ROOT / 'reports/qwen7b_report_context.json'
QWEN_REPORT_MD = DRIVE_ROOT / 'reports/qwen7b_final_report.md'
QWEN_REPORT_METADATA = DRIVE_ROOT / 'reports/qwen7b_final_report.metadata.json'
LLM_EVAL_JSON = DRIVE_ROOT / 'reports/llm_eval_summary.json'
LLM_EVAL_MD = DRIVE_ROOT / 'reports/llm_eval_summary.md'
LATEST_METRICS_EVIDENCE = DRIVE_ROOT / 'docs/task12_latest_metrics.md'
DEMO_DISPLAY_SMOKE_JSON = DRIVE_ROOT / 'reports/final_portfolio_demo_display_smoke.json'
BACKUP_ROOT = DRIVE_ROOT / 'backups'

for key, value in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'YOLO_WEIGHTS': YOLO_WEIGHTS,
    'QWEN_ADAPTER_DIR': QWEN_ADAPTER_DIR,
    'CONTEXT_JSON': CONTEXT_JSON,
    'QWEN_REPORT_MD': QWEN_REPORT_MD,
    'QWEN_REPORT_METADATA': QWEN_REPORT_METADATA,
    'LLM_EVAL_JSON': LLM_EVAL_JSON,
    'LLM_EVAL_MD': LLM_EVAL_MD,
}.items():
    os.environ[key] = str(value)
    print(f'{key} = {value}')


In [ ]:
# CN: ??????????? Drive??????????adapter ? runs?
# EN: Sync latest lightweight repo code into Drive without touching heavy artifacts.
%cd /content
!apt-get update -qq && apt-get install -y -qq rsync
!rm -rf "$REPO_CLONE_DIR"
!git clone "$REPO_URL" "$REPO_CLONE_DIR"

for folder in ['src', 'scripts', 'tests', 'docs', 'configs', 'notebooks']:
    src = REPO_CLONE_DIR / folder
    dst = DRIVE_ROOT / folder
    dst.mkdir(parents=True, exist_ok=True)
    !rsync -a --delete "$src/" "$dst/"

for file_name in ['README.md', 'requirements-colab.txt', 'pyproject.toml']:
    src = REPO_CLONE_DIR / file_name
    if src.exists():
        !cp "$src" "$DRIVE_ROOT/$file_name"

print('Lightweight sync complete. / ?????????')


In [ ]:
# CN: ?? Colab ??? editable package?
# EN: Install Colab dependencies and editable package.
%cd /content/drive/MyDrive/CarDD_YOLO11
%pip install -q -U pip
%pip install -q gdown
%pip install -q -e ".[vision,llm,service,dev]"


In [ ]:
# CN: ???? Drive ????????
# EN: Check required Drive artifacts and latest metrics.
import json
from pathlib import Path

required = {
    'YOLO weights': YOLO_WEIGHTS,
    'adapter_config.json': QWEN_ADAPTER_DIR / 'adapter_config.json',
    'adapter_model.safetensors': QWEN_ADAPTER_DIR / 'adapter_model.safetensors',
    'tokenizer_config.json': QWEN_ADAPTER_DIR / 'tokenizer_config.json',
    'tokenizer.json': QWEN_ADAPTER_DIR / 'tokenizer.json',
    'test metrics': DRIVE_ROOT / 'reports/yolo11n_seg_test_metrics.json',
    'run summary': DRIVE_ROOT / 'reports/yolo11n_seg_run_summary.json',
}
missing = []
for label, item in required.items():
    if item.exists():
        print(f'OK {label}: {item}')
    else:
        print(f'MISSING {label}: {item}')
        missing.append(str(item))
assert not missing, missing

metrics = json.loads((DRIVE_ROOT / 'reports/yolo11n_seg_test_metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
assert round(float(metrics['metrics/mAP50(B)']), 4) == 0.6746
assert round(float(metrics['metrics/mAP50(M)']), 4) == 0.6712


In [ ]:
# CN: ?????????????????
# EN: Package-level verification. Must pass before continuing.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m compileall src scripts
!python -m pytest tests -q -p no:cacheprovider


In [ ]:
# CN: ????????? LLM/RAG eval?
# EN: Generate project report and run LLM/RAG eval.
%cd /content/drive/MyDrive/CarDD_YOLO11
import json
import subprocess
import sys
from pathlib import Path

!python -m vehicle_damage_pipeline.report.build_context --drive-root "$DRIVE_ROOT" --output "$CONTEXT_JSON"
!python -m vehicle_damage_pipeline.report.generate --context "$CONTEXT_JSON" --output "$QWEN_REPORT_MD" --language Chinese --backend qwen --adapter-dir "$QWEN_ADAPTER_DIR" --metadata-output "$QWEN_REPORT_METADATA"

context = json.loads(Path(CONTEXT_JSON).read_text(encoding='utf-8'))
metadata = json.loads(Path(QWEN_REPORT_METADATA).read_text(encoding='utf-8'))
print(json.dumps(metadata, ensure_ascii=False, indent=2)[:5000])
assert metadata['final_validation']['passed'] is True
if metadata.get('backend') == 'template':
    assert metadata.get('fallback_reason'), metadata

box_map50 = float(context['test_metrics']['metrics/mAP50(B)'])
mask_map50 = float(context['test_metrics']['metrics/mAP50(M)'])
LATEST_METRICS_EVIDENCE.parent.mkdir(parents=True, exist_ok=True)
LATEST_METRICS_EVIDENCE.write_text(
    f'''# Task 1+2 Latest Metrics Evidence

Dataset: CarDD
Model: YOLO11n-seg
Box mAP50: {box_map50:.4f}
Mask mAP50: {mask_map50:.4f}
Demo label: scratch
Artifact: best.pt
''',
    encoding='utf-8',
)

cmd = [
    sys.executable,
    '-m',
    'vehicle_damage_pipeline.eval.run_llm_eval',
    '--context', str(CONTEXT_JSON),
    '--report', str(QWEN_REPORT_MD),
    '--knowledge-root', str(DRIVE_ROOT),
    '--output-json', str(LLM_EVAL_JSON),
    '--output-markdown', str(LLM_EVAL_MD),
]
subprocess.run(cmd, check=True)
summary = json.loads(Path(LLM_EVAL_JSON).read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2)[:5000])
assert summary['passed'] is True
assert summary['report_eval']['passed'] is True
assert summary['retrieval_eval']['checks']['box_map50']['hit'] is True
assert summary['retrieval_eval']['checks']['mask_map50']['hit'] is True


In [ ]:
# CN: ?????? smoke test?Qwen ?????? fallback ??????
# EN: Single-image report safety smoke test: hallucinated Qwen output must fallback.
from vehicle_damage_pipeline.service.predictor import generate_assessment_report, evaluate_assessment_report

dent_prediction = {
    'image_name': '000033.jpg',
    'detections': [
        {
            'class_id': 0,
            'class_name': 'dent',
            'confidence': 0.68,
            'bbox_xyxy': [282.0, 244.1, 621.1, 529.7],
            'mask_polygon': [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]],
        }
    ],
    'summary': '1 dent',
}
hallucinated = '???????????????????????????????????'
report = generate_assessment_report(
    dent_prediction,
    backend='qwen',
    adapter_dir=QWEN_ADAPTER_DIR,
    qwen_generate_fn=lambda **_: hallucinated,
)
validation = evaluate_assessment_report(dent_prediction, report)
print(report)
print(json.dumps(validation, ensure_ascii=False, indent=2))
assert validation['passed'] is True
for phrase in ['??????', '????????', '????', '??', '????']:
    assert phrase not in report

empty_report = generate_assessment_report({'detections': []}, backend='template')
assert '????' in empty_report
assert '????' not in empty_report
print(empty_report)


In [ ]:
# CN: demo ?? smoke test?????/??????? polygon?debug JSON ???????
# EN: Demo display smoke test: public summary/table hide full polygon, debug JSON keeps it.
from vehicle_damage_pipeline.service.display import build_public_prediction_summary, build_detection_table, build_debug_prediction_json
import json

public_summary = build_public_prediction_summary(dent_prediction)
detection_table = build_detection_table(dent_prediction)
debug_json = build_debug_prediction_json(dent_prediction)

public_text = json.dumps(public_summary, ensure_ascii=False)
assert 'mask_polygon' not in public_text
assert public_summary['detections'][0]['mask_points'] == 3
assert detection_table[0]['class_name'] == 'dent'
assert detection_table[0]['confidence'] == 0.68
assert detection_table[0]['mask_points'] == 3
assert json.loads(debug_json)['detections'][0]['mask_polygon'] == [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]

DEMO_DISPLAY_SMOKE_JSON.parent.mkdir(parents=True, exist_ok=True)
DEMO_DISPLAY_SMOKE_JSON.write_text(
    json.dumps(
        {
            'public_summary': public_summary,
            'detection_table': detection_table,
            'debug_json_has_mask_polygon': 'mask_polygon' in debug_json,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)
print(json.dumps(json.loads(DEMO_DISPLAY_SMOKE_JSON.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))


## Optional Public Gradio Launch / ???? Demo ??

The validation notebook does not launch a long-running Gradio app by default. Run the next cell manually only when you want a temporary public demo link.

?? notebook ??????????? Gradio????????? demo ???????????? cell?

In [ ]:
# Optional: launch public Gradio demo manually.
# %cd /content/drive/MyDrive/CarDD_YOLO11
# !python scripts/colab_gradio_public_demo.py --weights "$YOLO_WEIGHTS" --output-dir "$DRIVE_ROOT/runs/gradio_predict" --sample-dir "$DRIVE_ROOT/demo_upload_samples" --adapter-dir "$QWEN_ADAPTER_DIR" --server-port 7860


In [ ]:
# CN: ??????????????????????ONNX ? adapter?
# EN: Back up final portfolio validation evidence. Data, weights, ONNX, and adapters are not copied.
from datetime import datetime
import json
import shutil
import zipfile

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir = BACKUP_ROOT / f'final_portfolio_validation_{stamp}'
backup_dir.mkdir(parents=True, exist_ok=True)

items = [
    CONTEXT_JSON,
    QWEN_REPORT_MD,
    QWEN_REPORT_METADATA,
    LLM_EVAL_JSON,
    LLM_EVAL_MD,
    LATEST_METRICS_EVIDENCE,
    DEMO_DISPLAY_SMOKE_JSON,
    DRIVE_ROOT / 'notebooks/07_colab_final_portfolio_validation.ipynb',
]
manifest = {'created_at': datetime.now().isoformat(timespec='seconds'), 'entries': [], 'missing': []}
for src in items:
    src = Path(src)
    if src.exists() and src.is_file():
        dst = backup_dir / src.relative_to(DRIVE_ROOT)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        manifest['entries'].append({'source': str(src), 'backup': str(dst), 'size_bytes': src.stat().st_size})
    else:
        manifest['missing'].append(str(src))

manifest_path = backup_dir / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
zip_path = backup_dir.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file in backup_dir.rglob('*'):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(backup_dir.parent))

print('Backup dir / ????:', backup_dir)
print('Backup zip / ?????:', zip_path)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['missing'] == []


## Result / ??

If all cells above pass, the project has a fresh Colab-side final validation evidence bundle. The public README should still avoid linking the private Drive root; only curated lightweight artifacts should be shared publicly.

???? cell ???????????? Colab ??????????? README ??????? Drive ??????????????????